# Laptop Price Predictor — Part 1 of 4
### Import Library · Data Loading · Feature Extraction · Data Info

**Project goal:** predict a laptop's **price** from its specification (brand, processor, RAM, storage, screen, …).

The assignment has 20 steps. They are split across four notebooks, in the same order as the assignment document.
Run the parts in order: each part saves a CSV file that the next part loads.

| Part | Notebook | Assignment steps |
|---|---|---|
| **1** | **`Part1_Data_Loading_and_Feature_Extraction.ipynb`** ← you are here | **1 Import Library · 2 Data Loading · 3 Feature Extraction · 4 Data Info** |
| 2 | `Part2_Missing_Data_and_Duplicates.ipynb` | 5 Checking missing data · 6 Missing Data Treatment · 7 Duplicate Checking · 8 Duplicate Removing |
| 3 | `Part3_Visualization_and_Feature_Engineering.ipynb` | 9 Data Visualization · 10 Correlation with target · 11 Features and target separation · 12 Feature Selection · 13 Feature Scaling · 14 PCA · 15 Encoding |
| 4 | `Part4_Model_Training_and_Cross_Validation.ipynb` | 16 Train Test Split · 17 Model Preparation · 18 Model Accuracy · 19 Cross_val function · 20 cross_val Accuracy |

**Input:** `Cleaned_Laptop_data.csv` → **Output:** `laptop_part1_features.csv`

**Running in Google Colab:** *Runtime → Run all*. If `Cleaned_Laptop_data.csv` is not already in the Colab
file panel, the Data Loading cell asks you to upload it. At the end, the output CSV is downloaded
automatically; upload it into Part 2.

## Step 1: Import Library

In [1]:
import os
import re

import numpy as np
import pandas as pd

try:
    from google.colab import files  # this import only works inside Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)


def find_file(name):
    """Find a data file next to the notebook, in a data/ folder or in Colab's /content.
    In Colab, if the file is missing, ask for an upload."""
    for folder in [".", "data", "/content", "/content/data"]:
        path = os.path.join(folder, name)
        if os.path.exists(path):
            return path
    if IN_COLAB:
        print(f"'{name}' was not found - please upload it.")
        uploaded = files.upload()
        return next(iter(uploaded))  # Colab saves the upload in /content
    raise FileNotFoundError(f"Put '{name}' next to this notebook (or in a data/ folder) and run again.")


def save_output(frame, name):
    """Save a CSV for the next part. In Colab the file is also downloaded so you can upload it there."""
    folder = "data" if os.path.isdir("data") else "."
    path = os.path.join(folder, name)
    frame.to_csv(path, index=False)
    print(f"Saved {path}  ->  {frame.shape[0]} rows x {frame.shape[1]} columns")
    if IN_COLAB:
        files.download(path)


print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("Running in Google Colab:", IN_COLAB)

numpy : 2.4.6
pandas: 3.0.6
Running in Google Colab: False


## Step 2: Data Loading

In [2]:
data_path = find_file("Cleaned_Laptop_data.csv")
raw = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Shape :", raw.shape, "(rows, columns)")
raw.head(10)

Loaded: data/Cleaned_Laptop_data.csv
Shape : (845, 18) (rows, columns)


,brand,model,processor_brand,processor_Name,processor_gnrtn,ram_gb,Apps,ssd,hdd,os,os_bit,graphic_card_gb,weight,display_size,warranty,Touchscreen,msoffice,Price
0,ASUS,Celeron,Intel,5,Missing,4.0,Cooling,0,1024,Windows,64,0,Casual,15.6,1,No,No,23990.0
1,ASUS,VivoBook,Intel,5,10th,4.0,DDR3,512,0,Windows,64,0,Casual,15.6,1,No,No,37990.0
2,ASUS,Vivobook,Intel,A6-9225 Processor,10th,4.0,DDR3,0,1024,Windows,64,0,Casual,14.1,1,No,No,NaN
3,HP,Core,Intel,APU Dual,11th,4.0,DDR3,512,0,Windows,64,0,ThinNlight,15.6,1,No,Yes,54990.0
4,HP,Core,Intel,APU Dual,11th,4.0,DDR3,512,0,Windows,64,0,ThinNlight,15.6,0,No,No,54990.0
5,Lenovo,IdeaPad,Intel,APU Dual,10th,4.0,DDR3,0,1024,Windows,64,0,ThinNlight,15.6,1,No,Yes,35990.0
6,HP,15s,AMD,APU Dual,Missing,4.0,DDR3,256,1024,Windows,64,0,Casual,15.6,1,No,Yes,41999.0
7,ASUS,Core,Intel,APU Dual,11th,4.0,DDR3,256,0,Windows,64,0,ThinNlight,14,0,No,No,34429.0
8,DELL,Vostro,Intel,APU Dual,10th,4.0,DDR3,256,1024,Windows,64,0,ThinNlight,15.6,1,No,Yes,41490.0
9,Lenovo,IdeaPad,Intel,APU Dual,10th,4.0,DDR3,256,0,Windows,32,0,ThinNlight,15.6,2,No,Yes,37990.0


In [3]:
raw.tail(5)

,brand,model,processor_brand,processor_Name,processor_gnrtn,ram_gb,Apps,ssd,hdd,os,os_bit,graphic_card_gb,weight,display_size,warranty,Touchscreen,msoffice,Price
840,DELL,Inspiron,AMD,Ryzen 9,Missing,16.0,Unified,512,0,Windows,64,0,ThinNlight,15.6,0,No,No,42090.0
841,HP,Pavilion,AMD,Ryzen 9,Missing,16.0,Unified,512,0,Windows,64,4,Gaming,15.6,1,No,Yes,64990.0
842,ASUS,EeeBook,Intel,Ryzen 9,Missing,16.0,Unified,0,0,Windows,64,0,Casual,0,0,No,No,22990.0
843,HP,Pavilion,AMD,Slot for,Missing,16.0,Unified,512,0,Windows,64,2,Casual,14,1,No,Yes,79990.0
844,Lenovo,Ideapad,Intel,Snapdragon 7c,10th,16.0,Unified,512,0,Windows,64,2,Casual,14,1,Yes,Yes,86999.0


## Step 3: Feature Extraction

Feature extraction turns raw columns into clean features a model can use. First, a look at what each column
really contains:

In [4]:
pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "unique values": raw.nunique(),
    "example values": [raw[c].dropna().unique()[:6].tolist() for c in raw.columns],
})

,dtype,unique values,example values
brand,str,21,"[ASUS, HP, Lenovo, DELL, APPLE, acer]"
model,str,116,"[Celeron, VivoBook, Vivobook, Core, IdeaPad, 15s]"
processor_brand,str,10,"[Intel, AMD, Apple, MediaTek, NVIDIA, Qualcomm]"
processor_Name,str,34,"[5, A6-9225 Processor, APU Dual, Athlon Dual, ..."
processor_gnrtn,str,8,"[Missing, 10th, 11th, 9th, 8th, 7th]"
ram_gb,float64,5,"[4.0, 5.0, 8.0, 15.6, 16.0]"
Apps,str,14,"[Cooling, DDR3, DDR4, DDR5, Full, GTX]"
ssd,int64,8,"[0, 512, 256, 1024, 2048, 128]"
hdd,int64,3,"[1024, 0, 512]"
os,str,3,"[Windows, Mac, Missing]"


What the table shows, and what we extract from it:

| # | Problem in the raw data | Feature extraction |
|---|---|---|
| 3.1 | `Apps` holds RAM types (DDR4, LPDDR4X…), `weight` holds laptop categories (Casual, ThinNlight, Gaming); column names mix upper and lower case | Rename `Apps` → `ram_type`, `weight` → `laptop_type`; make every column name lower case |
| 3.2 | The same brand or model is written differently (`Lenovo` / `lenovo`, `VivoBook` / `Vivobook`) | Standardise the text: `brand` in upper case, `model` in lower case |
| 3.3 | `processor_gnrtn` is text such as `"11th"` | Extract the number → `processor_gen` = 11 |
| 3.4 | `processor_name` has 34 messy values (`Core i5`, `i5`, `Ryzen 7`, `M1 Pro`, `GeForce RTX`…) | Group them into a clean `cpu_family` (Intel Core i3/i5/i7/i9, AMD Ryzen 3/5/7/9, Apple M1, Entry level) |
| 3.5 | `display_size` is stored as text and contains words like `All`, `6th` | Convert to a number → `display_inch` |
| 3.6 | Storage is split across `ssd` and `hdd` | New feature `total_storage_gb` = `ssd` + `hdd` |

### 3.1 Rename columns

In [5]:
df = raw.copy()
df = df.rename(columns={"Apps": "ram_type", "weight": "laptop_type"})
df.columns = df.columns.str.lower()

print("Before:", raw.columns.tolist())
print("After :", df.columns.tolist())

Before: ['brand', 'model', 'processor_brand', 'processor_Name', 'processor_gnrtn', 'ram_gb', 'Apps', 'ssd', 'hdd', 'os', 'os_bit', 'graphic_card_gb', 'weight', 'display_size', 'warranty', 'Touchscreen', 'msoffice', 'Price']
After : ['brand', 'model', 'processor_brand', 'processor_name', 'processor_gnrtn', 'ram_gb', 'ram_type', 'ssd', 'hdd', 'os', 'os_bit', 'graphic_card_gb', 'laptop_type', 'display_size', 'warranty', 'touchscreen', 'msoffice', 'price']


### 3.2 Standardise brand and model text

In [6]:
df["brand"] = df["brand"].str.strip().str.upper()
df["model"] = df["model"].str.strip().str.lower()

print("Unique brands:", raw["brand"].nunique(), "->", df["brand"].nunique())
print("Unique models:", raw["model"].nunique(), "->", df["model"].nunique())
print()
print(df["brand"].value_counts().to_string())

Unique brands: 21 -> 20
Unique models: 116 -> 106

brand
ASUS         246
DELL         147
LENOVO       141
HP           133
ACER          52
MSI           51
APPLE         27
AVITA         10
LG             5
VAIO           5
REALME         4
NOKIA          4
INFINIX        4
ALIENWARE      4
REDMIBOOK      3
MICROSOFT      3
MI             2
SMARTRON       2
SAMSUNG        1
IBALL          1


### 3.3 Processor generation: text → number

In [7]:
df["processor_gen"] = pd.to_numeric(df["processor_gnrtn"].str.extract(r"(\d+)")[0], errors="coerce")

(df.groupby("processor_gnrtn")
   .agg(processor_gen=("processor_gen", "first"), laptops=("processor_gen", "size"))
   .sort_values("processor_gen"))

,processor_gen,laptops
processor_gnrtn,,
4th,4.0,1
7th,7.0,9
8th,8.0,27
9th,9.0,5
10th,10.0,162
11th,11.0,334
12th,12.0,3
Missing,NaN,304


`"Missing"` has no number, so it becomes `NaN` (an empty value). Part 2 deals with it.

### 3.4 Processor name → CPU family

In [8]:
ENTRY_LEVEL_WORDS = ["celeron", "pentium", "athlon", "apu", "dual core", "quad", "hexa",
                     "mediatek", "snapdragon", "a6-9225"]


def get_cpu_family(name):
    """Map a raw processor name to a clean CPU family. Names that are not a processor give NaN."""
    text = str(name).lower()
    intel = re.search(r"\bi([3579])\b", text)          # "core i5", "i5"
    if intel:
        return "Intel Core i" + intel.group(1)
    ryzen = re.search(r"ryzen\s*([3579])", text)       # "ryzen 7"
    if ryzen:
        return "AMD Ryzen " + ryzen.group(1)
    if "m1" in text:                                   # "m1 processor", "m1 pro", "m1 max"
        return "Apple M1"
    if any(word in text for word in ENTRY_LEVEL_WORDS):
        return "Entry level"
    return np.nan                                      # e.g. "GeForce RTX", "Genuine Windows"


df["cpu_family"] = df["processor_name"].apply(get_cpu_family)

(df.groupby("processor_name")
   .agg(cpu_family=("cpu_family", "first"), laptops=("cpu_family", "size"))
   .sort_values("laptops", ascending=False))

,cpu_family,laptops
processor_name,,
Core i5,Intel Core i5,274
Core i3,Intel Core i3,158
Core i7,Intel Core i7,103
Ryzen 5,AMD Ryzen 5,82
Ryzen 7,AMD Ryzen 7,57
Ryzen 9,AMD Ryzen 9,26
Ryzen 3,AMD Ryzen 3,24
Celeron Dual,Entry level,20
M1 Processor,Apple M1,17


In [9]:
print("processor_name:", df["processor_name"].nunique(), "values  ->  cpu_family:",
      df["cpu_family"].nunique(), "values\n")
print(df["cpu_family"].value_counts(dropna=False).to_string())

processor_name: 34 values  ->  cpu_family: 10 values

cpu_family
Intel Core i5    280
Intel Core i3    158
Intel Core i7    116
AMD Ryzen 5       82
AMD Ryzen 7       57
Entry level       54
AMD Ryzen 9       26
AMD Ryzen 3       24
Apple M1          23
NaN               18
Intel Core i9      7


Names such as `GeForce RTX` (a graphics card), `Genuine Windows` or `GB SSD` are not processors: they were put in
the wrong column when the data was scraped. They become `NaN` here and are treated in Part 2.

### 3.5 Display size: text → number

In [10]:
df["display_inch"] = pd.to_numeric(df["display_size"], errors="coerce")

print(df["display_inch"].value_counts(dropna=False).sort_index().to_string())

display_inch
0.00      58
12.00      1
12.20      2
13.00      7
13.30     77
13.40      7
13.50      6
14.00    268
14.10      7
14.20      3
14.90      1
14.96      9
15.00      6
15.30      1
15.60    358
16.00      8
16.10      7
16.20      3
17.30     13
NaN        3


Text that is not a size (`All`, `6th`, `8th`) became `NaN`. A screen of `0` inches is impossible; Part 2 counts
it as missing.

### 3.6 New feature: total storage

In [11]:
df["total_storage_gb"] = df["ssd"] + df["hdd"]

df[["ssd", "hdd", "total_storage_gb"]].value_counts().head(10)

ssd   hdd   total_storage_gb
512   0     512                 400
256   0     256                 117
1024  0     1024                109
256   1024  1280                 71
0     1024  1024                 69
      0     0                    60
128   0     128                   9
512   1024  1536                  3
2048  0     2048                  2
512   512   1024                  2
Name: count, dtype: int64

### 3.7 Drop the raw text columns that now have a numeric version

`processor_gnrtn` is replaced by `processor_gen`, and `display_size` by `display_inch`. `processor_name` stays for
now (`cpu_family` is a grouping of it); Part 3 decides which features the model keeps.

In [12]:
df = df.drop(columns=["processor_gnrtn", "display_size"])

print("Shape after feature extraction:", df.shape)
df.head()

Shape after feature extraction: (845, 20)


,brand,model,processor_brand,processor_name,ram_gb,ram_type,ssd,hdd,os,os_bit,graphic_card_gb,laptop_type,warranty,touchscreen,msoffice,price,processor_gen,cpu_family,display_inch,total_storage_gb
0,ASUS,celeron,Intel,5,4.0,Cooling,0,1024,Windows,64,0,Casual,1,No,No,23990.0,NaN,NaN,15.6,1024
1,ASUS,vivobook,Intel,5,4.0,DDR3,512,0,Windows,64,0,Casual,1,No,No,37990.0,10.0,NaN,15.6,512
2,ASUS,vivobook,Intel,A6-9225 Processor,4.0,DDR3,0,1024,Windows,64,0,Casual,1,No,No,NaN,10.0,Entry level,14.1,1024
3,HP,core,Intel,APU Dual,4.0,DDR3,512,0,Windows,64,0,ThinNlight,1,No,Yes,54990.0,11.0,Entry level,15.6,512
4,HP,core,Intel,APU Dual,4.0,DDR3,512,0,Windows,64,0,ThinNlight,0,No,No,54990.0,11.0,Entry level,15.6,512


## Step 4: Data Info

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 845 entries, 0 to 844
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   brand             845 non-null    str    
 1   model             845 non-null    str    
 2   processor_brand   845 non-null    str    
 3   processor_name    845 non-null    str    
 4   ram_gb            845 non-null    float64
 5   ram_type          845 non-null    str    
 6   ssd               845 non-null    int64  
 7   hdd               845 non-null    int64  
 8   os                845 non-null    str    
 9   os_bit            845 non-null    int64  
 10  graphic_card_gb   845 non-null    int64  
 11  laptop_type       845 non-null    str    
 12  warranty          845 non-null    int64  
 13  touchscreen       845 non-null    str    
 14  msoffice          845 non-null    str    
 15  price             842 non-null    float64
 16  processor_gen     541 non-null    float64
 17  cpu_fami

In [14]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
ram_gb,845.0,9.77,4.11,4.0,8.0,8.0,16.0,16.0
ssd,845.0,444.48,315.17,0.0,256.0,512.0,512.0,3072.0
hdd,845.0,175.72,385.50,0.0,0.0,0.0,0.0,1024.0
os_bit,845.0,59.34,11.29,32.0,64.0,64.0,64.0,64.0
graphic_card_gb,845.0,1.23,2.08,0.0,0.0,0.0,2.0,8.0
warranty,845.0,0.71,0.61,0.0,0.0,1.0,1.0,3.0
price,842.0,77355.59,47077.04,13990.0,46322.5,63990.0,91740.0,441990.0
processor_gen,541.0,10.46,0.93,4.0,10.0,11.0,11.0,12.0
display_inch,842.0,13.75,3.86,0.0,14.0,14.0,15.6,17.3
total_storage_gb,845.0,620.19,375.42,0.0,512.0,512.0,1024.0,3072.0


In [15]:
df.describe(exclude="number").T

,count,unique,top,freq
brand,845,20,ASUS,246
model,845,106,core,87
processor_brand,845,10,Intel,599
processor_name,845,34,Core i5,274
ram_type,845,14,DDR4,717
os,845,3,Windows,754
laptop_type,845,3,Casual,534
touchscreen,845,2,No,744
msoffice,845,2,No,564
cpu_family,827,10,Intel Core i5,280


In [16]:
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non-null": df.notna().sum(),
    "NaN": df.isna().sum(),
    "unique": df.nunique(),
}).sort_values("NaN", ascending=False)

,dtype,non-null,NaN,unique
processor_gen,float64,541,304,7
cpu_family,str,827,18,10
price,float64,842,3,403
display_inch,float64,842,3,19
processor_name,str,845,0,34
processor_brand,str,845,0,10
model,str,845,0,106
brand,str,845,0,20
ram_gb,float64,845,0,5
ram_type,str,845,0,14


In [17]:
for col in ["os", "ram_type", "laptop_type", "processor_brand", "touchscreen", "msoffice"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_string(), end="\n\n")

--- os ---
os
Windows    754
Missing     64
Mac         27

--- ram_type ---
ram_type
DDR4        717
LPDDR4X      65
LPDDR3       14
DDR3         12
LPDDR4       11
DDR5          7
Full          6
Unified       6
GTX           2
Cooling       1
i5            1
LED           1
Master        1
ram_type      1

--- laptop_type ---
laptop_type
Casual        534
ThinNlight    273
Gaming         38

--- processor_brand ---
processor_brand
Intel            599
AMD              205
Apple             23
NVIDIA             7
MediaTek           3
Pre-installed      3
First              2
Qualcomm           1
512                1
M.2                1

--- touchscreen ---
touchscreen
No     744
Yes    101

--- msoffice ---
msoffice
No     564
Yes    281



### What Data Info tells us

- **845 laptops × 20 columns** after feature extraction: 10 numeric and 10 text columns.
- **Visible missing values (`NaN`)**: `processor_gen` 304 (the generation was written as `"Missing"`),
  `cpu_family` 18 (the processor name was not a processor), `price` 3 and `display_inch` 3.
- **Hidden missing values** that look like real data: `os` = `"Missing"` (64 laptops), a screen size of 0 inches
  (58), RAM of 15.6 GB or 5 GB (7), non-CPU makers in `processor_brand` (NVIDIA, M.2, 512…), non-RAM values in
  `ram_type` (GTX, LED, Cooling…), and 60 laptops with 0 GB of storage.
- **Price** ranges from 13,990 to 441,990. The mean (77,356) is well above the median (63,990), so a few very
  expensive laptops pull the average up.
- Most laptops are ASUS (246), run Windows (754), use DDR4 RAM (717) and are `Casual` laptops (534).

Part 2 handles all of these problems.

In [18]:
save_output(df, "laptop_part1_features.csv")

Saved data/laptop_part1_features.csv  ->  845 rows x 20 columns


**Next:** open `Part2_Missing_Data_and_Duplicates.ipynb`.